In [1]:
import pandas as pd
from fastparquet import write
from fastparquet import ParquetFile
from pathlib import Path
import numpy as np
import os

In [2]:
data_pipeline = "features"
input_pipeline = "baseline"

In [3]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [4]:
experiment_path = Path(data_path) / f"{data_pipeline}"
experiment_path.mkdir(parents=True, exist_ok=True)

In [5]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [6]:
X = ParquetFile(Path(data_path) / f"{input_pipeline}/train.parq").to_pandas()
X_test = ParquetFile(Path(data_path) / f"{input_pipeline}/test.parq").to_pandas()

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   age                      662440 non-null  float64 
 1   daily_screen_time_hours  595515 non-null  float64 
 2   social_media_hours       557374 non-null  float64 
 3   gaming_hours             564548 non-null  float64 
 4   work_study_hours         639851 non-null  float64 
 5   sleep_hours              646889 non-null  float64 
 6   notifications_per_day    623785 non-null  float64 
 7   app_opens_per_day        610659 non-null  float64 
 8   weekend_screen_time      579306 non-null  float64 
 9   gender                   662335 non-null  category
 10  stress_level             636221 non-null  float64 
 11  academic_work_impact     647145 non-null  float64 
dtypes: category(1), float64(11)
memory usage: 58.7 MB


In [7]:
drop_columns = X.columns
for frame in [X, X_test]:
    frame['screen_sleep_ratio'] = frame['daily_screen_time_hours'] / frame['sleep_hours']
    frame['total_activity'] = (frame['social_media_hours'] + frame['gaming_hours'] + frame['work_study_hours'])

    frame['social_media_ratio'] = frame['social_media_hours'] / (frame['total_activity'] + 1)
    frame['gaming_ratio'] = frame['gaming_hours'] / (frame['total_activity'] + 1)
    frame['work_ratio'] = frame['work_study_hours'] / (frame['total_activity'] + 1)

    frame['daily_free_hours'] = 24 - (frame['daily_screen_time_hours'] + frame['sleep_hours'])
    frame["daily_extra_screen_time_hours"] = frame['daily_screen_time_hours'] - (frame['social_media_hours'] + frame['gaming_hours'] + frame['work_study_hours'])
    frame['weekend_extra_screen_time'] = frame['weekend_screen_time'] - frame['daily_screen_time_hours']
    frame['phone_activity'] = frame['notifications_per_day'] * frame['app_opens_per_day']
    frame['hourly_notifications_per_day'] = frame['notifications_per_day'] / frame['daily_screen_time_hours']
    frame['hourly_app_opens_per_day'] = frame['app_opens_per_day'] / frame['daily_screen_time_hours']

    frame['stress_daily_hours_impact'] = (frame['stress_level'] * frame['daily_screen_time_hours']).astype(np.float64)
    frame['stress_social_media_hours_impact'] = (frame['stress_level'] * frame['social_media_hours']).astype(np.float64)
    frame['stress_work_study_hours_impact'] = (frame['stress_level'] * frame['work_study_hours']).astype(np.float64)

    frame['stress_app_opens_impact'] =  (frame['stress_level'] * frame['app_opens_per_day']).astype(np.float64)
    frame['stress_notifications_impact'] =  (frame['stress_level'] * frame['notifications_per_day']).astype(np.float64)

    frame.drop(drop_columns, axis=1, inplace=True)

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 16 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   screen_sleep_ratio                559350 non-null  float64
 1   total_activity                    451825 non-null  float64
 2   social_media_ratio                451825 non-null  float64
 3   gaming_ratio                      451825 non-null  float64
 4   work_ratio                        451825 non-null  float64
 5   daily_free_hours                  559350 non-null  float64
 6   daily_extra_screen_time_hours     421427 non-null  float64
 7   weekend_extra_screen_time         517906 non-null  float64
 8   phone_activity                    568082 non-null  float64
 9   hourly_notifications_per_day      540721 non-null  float64
 10  hourly_app_opens_per_day          529819 non-null  float64
 11  stress_daily_hours_impact         550740 non-null  float64
 12 

In [ ]:
write(experiment_path / f"train.parq", X)
write(experiment_path / f"test.parq", X_test)